Download dataset from [here](https://www.kaggle.com/datasets/usharengaraju/indian-women-in-defense?utm_source=chatgpt.com&select=airway_metadata.csv)

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind

counts = pd.read_csv('airway_scaledcounts.csv')
metadata = pd.read_csv('airway_metadata.csv')

In [3]:
control_samples = metadata[metadata['dex'] == 'control']['id'].tolist()
treated_samples = metadata[metadata['dex'] == 'treated']['id'].tolist()

counts["Control_Mean"] = counts[control_samples].mean(axis=1)
counts["Treated_Mean"] = counts[treated_samples].mean(axis=1)

counts.head()

,ensgene,SRR1039508,SRR1039509,SRR1039512,SRR1039513,SRR1039516,SRR1039517,SRR1039520,SRR1039521,Control_Mean,Treated_Mean
0,ENSG00000000003,723.0,486.0,904.0,445.0,1170.0,1097.0,806.0,604.0,900.75,658.00
1,ENSG00000000005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00
2,ENSG00000000419,467.0,523.0,616.0,371.0,582.0,781.0,417.0,509.0,520.50,546.00
3,ENSG00000000457,347.0,258.0,364.0,237.0,318.0,447.0,330.0,324.0,339.75,316.50
4,ENSG00000000460,96.0,81.0,73.0,66.0,118.0,94.0,102.0,74.0,97.25,78.75


In [ ]:
counts["Log2FC"] = np.log2(
    (counts["Treated_Mean"] + 1) /
    (counts["Control_Mean"] + 1)
)
p_values = []

for i in range(len(counts)):
    control_values = counts.loc[i, control_samples]
    treated_values = counts.loc[i, treated_samples]

    _, p_value = ttest_ind(control_values, treated_values)
    p_values.append(p_value)
counts["p_value"] = p_values

C:\Users\athni\AppData\Roaming\Python\Python314\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
C:\Users\athni\AppData\Roaming\Python\Python314\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
C:\Users\athni\AppData\Roaming\Python\Python314\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
C:\Users\athni\AppData\Roaming\Python\Python314\site-packages

KeyboardInterrupt: 

In [ ]:

# Find differentially expressed genes
deg = counts[
    (abs(counts["Log2FC"]) > 1) &
    (counts["p_value"] < 0.05)
].copy()

# Label genes as upregulated or downregulated
deg["Regulation"] = np.where(
    deg["Log2FC"] > 0,
    "Upregulated",
    "Downregulated"
)

# Display results
print(deg[["ensgene", "Control_Mean", "Treated_Mean",
           "Log2FC", "p_value", "Regulation"]])

# Save results
deg.to_csv("differentially_expressed_genes.csv", index=False)